# Double Agent Dilemma — Baseline

## 1. Setup

You load the two pretrained models (ResNet18 and ViT-Tiny) below and, for every image,
produce two perturbation tensors. Scoring is run by the graders after you submit — your
only job is to build the perturbation files.


In [17]:
import sys
from pathlib import Path
import numpy as np
import torch
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
from tqdm.auto import tqdm
import os

### PLEASE DO NOT CHANGE THESE VARIABLES ###
TRAIN_SPLIT = os.environ.get("TRAIN_SPLIT", "public/train")
TEST_SPLIT = os.environ.get("TEST_SPLIT", "public/test_public")
LB_A_SPLIT = os.environ.get("TEST_SPLIT", "private/test_leaderboard_a")
LB_B_SPLIT = os.environ.get("TEST_SPLIT", "private/test_leaderboard_b")
MODELS_ROOT = Path(os.environ.get("MODELS_DIR", "models"))
DATASET_ROOT = Path(os.environ.get("DATA_DIR", "dataset"))

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


## 2. Load the data

Two splits are provided:

- `train` (100 images) — explore and tune here.
- `test_public.` (100 images) — evaluation and debugging.

For labels.json, each item is `{'idx': int, 'image_path': Path, 'label': int}`, where `label` is the true
ImageNet-1K class index (0–999). Images have **varying original resolutions**.

In [2]:
import json

def load_split(split: str):
    """Load a dataset split (e.g. 'train', 'test_public.').

    Reads from dataset/<split>/ if present, otherwise extracts dataset/<split>.tar.gz.
    """
    split_dir = DATASET_ROOT / split
    if not split_dir.exists():
        raise FileNotFoundError(
            f"Dataset split '{split}' not found in {DATASET_ROOT}. "
            "Please make sure to extract the dataset first."
        )

    labels_path = split_dir / "labels.json"
    if not labels_path.exists():
        raise FileNotFoundError(
            f"labels.json not found in {split_dir}."
        )

    with open(labels_path) as f:
        labels = json.load(f)

    images_dir = split_dir / "images"
    items = []
    for idx_str, label in sorted(labels.items(), key=lambda x: int(x[0])):
        idx = int(idx_str)
        image_path = images_dir / f"{idx:04d}.png"
        items.append({"idx": idx, "image_path": image_path, "label": label})

    return items

In [3]:
train_data = load_split(TRAIN_SPLIT)
test_data = load_split(TEST_SPLIT)
lb_a_data = load_split(LB_A_SPLIT)
lb_b_data = load_split(LB_B_SPLIT)

print(f"train : {len(train_data)} images")
print(f"test  : {len(test_data)} images")

sample = train_data[0]
img0 = Image.open(sample["image_path"]).convert("RGB")
print(f"\nExample item: idx={sample['idx']}  label={sample['label']}  size={img0.size} (W×H)")

train : 100 images
test  : 100 images

Example item: idx=0  label=160  size=(495, 500) (W×H)


In [4]:
len(lb_a_data), len(lb_b_data)

(100, 100)

## 3. The Image Processing

The evaluation transform is fixed and applied to `image + perturbation` (pixels clipped to
`[0, 1]`) before either model sees it:

```
Resize(256) → CenterCrop(224) → Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
```

Below we load both models and the exact preprocessing, then confirm both classify the clean
image correctly — the starting point every attack must break for exactly one model at a time.


In [5]:
import timm
import torch
from torchvision import models

resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
#resnet.load_state_dict(torch.load(MODELS_ROOT / "resnet18.pth", map_location="cpu"))
resnet = resnet.to(DEVICE).eval()

# custom_load=False is required: this cfg has custom_load=True (the augreg
# weights were originally JAX .npz) and on the file= path timm honours it,
# feeding our safetensors to np.load. Plain pretrained=True escapes this only
# because the hf-hub path checks for custom_load == 'hf' instead.
vit = timm.create_model(
    "vit_tiny_patch16_224",
    pretrained=True,
    pretrained_cfg_overlay=dict(
        #file=MODELS_ROOT / "vit_tiny_patch16_224.safetensors",
        custom_load=False,
    ),
).to(DEVICE).eval()

for p in list(resnet.parameters()) + list(vit.parameters()):
    p.requires_grad = False

_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(3, 1, 1)

def preprocess(img_tensor):
    """Raw [0,1] CxHxW tensor -> normalized 3x224x224 (matches the grader transform)."""
    img = TF.resize(img_tensor, size=256, interpolation=T.InterpolationMode.BILINEAR)
    img = TF.center_crop(img, output_size=224)
    return (img - _MEAN) / _STD

@torch.no_grad()
def predict(img_tensor):
    inp = preprocess(img_tensor).unsqueeze(0)
    return int(resnet(inp).argmax(1)), int(vit(inp).argmax(1))

img_t = TF.to_tensor(img0).to(DEVICE)
r_pred, v_pred = predict(img_t)
print(f"True label     : {sample['label']}")
print(f"ResNet18 (Rex) : {r_pred}  {'✓' if r_pred == sample['label'] else '✗'}")
print(f"ViT-Tiny (Vita): {v_pred}  {'✓' if v_pred == sample['label'] else '✗'}")

True label     : 160
ResNet18 (Rex) : 160  ✓
ViT-Tiny (Vita): 160  ✓


## 4. How you are scored


```
Pure Score = (Score_A + Score_B) / (2M)
```

Then the L2 size of your perturbations is folded in as a multiplicative **penalty factor**
`PF` — the smaller (fainter) your perturbations, the higher `PF` (up to a neutral `1.0`);
larger, more visible perturbations are penalised (down to `0.5`). The final score is

```
S_final = Pure Score × PF
```

Scoring is performed by the graders on the hidden evaluator; you only produce `preds`. The
goal is to maximise Success A and B **while keeping each perturbation as small as possible.**


## 5. Optimization-Based Attack

This baseline uses gradient-based optimization via AdamW to generate distinct, localized perturbations ($\delta_a$ and $\delta_b$) for each target model:

* **Dual Perturbation Optimization:** Rather than applying uniform noise, the attack independently optimizes two separate bounded noise tensors ($\delta_a \in [-0.003, 0.003]$ and $\delta_b \in [-0.003, 0.003]$) to keep the perturbation small.
* **Targeted Model Disagreement:** For Type A ($\delta_a$), the optimization maximizes cross-entropy loss against the original label for the Vision Transformer (ViT) while disregarding the ResNet model. For Type B ($\delta_b$), the objective is inverted to maximize loss against the ground-truth label for the ResNet model while degrading the ViT's prediction. It turns out there is little cross-model transferability of the adversarial attacks in this setup (i.e., attacks that fool ResNet doesn't usually fool the ViT, and vice versa) so it was okay to disregard the model that is supposed to be predicting correctly to get a decent score.
* **Dynamic Adjustment:** The optimization dynamically resets its weight decay standard mid-run if both models predict the exact same class by iteration 10, forcing the models toward conflicting classifications. Norms of the noise tensors can be kept small if the attack is easy to minimize the penalty factor, but regularizations are relaxed to maintain a good attack success rate if a greater perturbation is necessary.

The `Solution` class matches the expected pipeline contract: `run(data)` processes each image through this gradient-driven procedure and returns `(preds, targets)`, where `preds` contains pairs of `(delta_a, delta_b)` CPU tensors matching the original input dimension ($3 \times H \times W$).

In [ ]:
import torch
import torchvision.transforms as T
from PIL import Image
from tqdm import tqdm

import random
import numpy as np

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

EPSILON = 0.0001
import torch.nn as nn
class Solution:
    """Baseline solution: naive constant-noise perturbation."""

    def __init__(self):
        pass

    def _generate_naive_noise(self, img_tensor, resnet, vit):
        mean_rgb = img_tensor.mean()
        label = predict(img_tensor.to(DEVICE))[0]
        label = torch.tensor([label]).to(DEVICE)
        noise_a = nn.Parameter(data=torch.randn_like(img_tensor).to(DEVICE) * 0.001, requires_grad=True)
        noise_a.requires_grad = True
        optimizer = torch.optim.AdamW([noise_a], lr=0.001, weight_decay=1e+03)
        criterion = nn.CrossEntropyLoss()
        resnet.eval()
        vit.eval()
        for item in range(25):
            optimizer.zero_grad()
            adv_a = torch.clamp(img_tensor.to(DEVICE) + torch.clamp(noise_a, -0.003, 0.003), 0, 1)
            inp_a = preprocess(adv_a).unsqueeze(0)
            pred_a = resnet(inp_a)
            pred_b = vit(inp_a)
            loss = 0 * criterion(pred_a, label) - criterion(pred_b, label)
            #print(noise.grad)
            loss.backward()
            optimizer.step()
            #print(pred_a[0].argmax(), pred_b[0].argmax())
            #print(loss)
            if item == 10 and pred_a[0].argmax() == pred_b[0].argmax():
                optimizer = torch.optim.AdamW([noise_a], lr=0.001)
        noise_a.data = torch.clamp(noise_a.data, -0.003, 0.003)
        noise_b = nn.Parameter(data=torch.randn_like(img_tensor).to(DEVICE) * 0.001, requires_grad=True)
        noise_b.requires_grad = True
        optimizer = torch.optim.AdamW([noise_b], lr=0.001, weight_decay=1e+03)
        criterion = nn.CrossEntropyLoss()
        for item in range(25):
            optimizer.zero_grad()
            adv_b = torch.clamp(img_tensor.to(DEVICE) + torch.clamp(noise_b, -0.003, 0.003), 0, 1)
            inp_b = preprocess(adv_b).unsqueeze(0)
            pred_a = resnet(inp_b)
            pred_b = vit(inp_b)
            loss = 0 * criterion(pred_b, label) - criterion(pred_a, label)
            #print(noise.grad)
            loss.backward()
            optimizer.step()
            #print(label, pred_a[0].argmax(), pred_b[0].argmax())
            #print(criterion(pred_b, label), criterion(pred_a, label))
            if item == 10 and pred_a[0].argmax() == pred_b[0].argmax():
                optimizer = torch.optim.AdamW([noise_b], lr=0.001)
        noise_b.data = torch.clamp(noise_b.data, -0.003, 0.003)
        return noise_a.data, noise_b.data #torch.full_like(img_tensor, mean_rgb + EPSILON)

    def run(self, data):
        """Return (preds, targets).

        preds:   list of (delta_a, delta_b) tensors
        targets: data (list of {'idx', 'image_path', 'label'})
        """
        preds = []
        for item in tqdm(data, desc="Generating naive perturbations"):
            img = Image.open(item["image_path"]).convert("RGB")
            img_tensor = T.functional.to_tensor(img)
            deltaa, deltab = self._generate_naive_noise(img_tensor, resnet, vit)
            preds.append((deltaa, deltab))
        return preds, data


## 6. Run

We first try a small slice of the `train` split to see the workflow end-to-end, then run the
full `test` split below.


In [7]:
solver = Solution()

# Quick smoke test on a handful of images.
subset = train_data[:20]
preds, targets = solver.run(subset)

Generating naive perturbations: 100%|██████████| 20/20 [00:19<00:00,  1.01it/s]


Then run on the full `test` split to generate a perturbation for every image.


In [8]:
test_preds, test_targets = solver.run(test_data)

Generating naive perturbations: 100%|██████████| 100/100 [01:35<00:00,  1.05it/s]


## 7. Check the submission

Before zipping, sanity-check that the perturbation files are well-formed. `check()`
compares a directory of `.pt` files against the original images and verifies:

1. **Count** — one `_a.pt` and one `_b.pt` per image, and no stray non-`.pt` files.
2. **Naming** — indices `{idx}_a.pt` / `{idx}_b.pt` exactly cover the image indices.
3. **Loadable** — every `.pt` loads with `weights_only=True` and is a tensor.
4. **Shape** — each tensor matches its raw image's `(3, H, W)`.

It returns `(ok, report)`. Point it at a folder of `.pt` files and the split's
`images/` directory, e.g. `check("submission_pt/", "dataset/<split>/images/")`.


In [9]:
import os
from pathlib import Path
from PIL import Image
import torch


def check(submission_dir, images_dir):
    """Check perturbation files against original images.

    Parameters
    ----------
    submission_dir : str or Path
        Directory containing flat-listed .pt files named {idx}_a.pt and {idx}_b.pt.
    images_dir : str or Path
        Directory containing original .png images named {idx:04d}.png.

    Returns
    -------
    ok : bool
    report : str
    """
    sub = Path(submission_dir)
    img = Path(images_dir)
    ok_parts = []
    issues = []

    if not sub.is_dir():
        return False, f"ERROR: {sub} is not a directory."
    if not img.is_dir():
        return False, f"ERROR: {img} is not a directory."

    # -- Get expected indices from image files --
    png_files = sorted(img.glob("*.png"))
    if not png_files:
        return False, f"ERROR: No .png files found in {img}"

    expected_indices = []
    for pth in png_files:
        try:
            expected_indices.append(int(pth.stem))
        except ValueError:
            issues.append(f"Bad image filename: {pth.name}")

    n = len(expected_indices)
    ok_parts.append(f"Images found: {n} ({img})")

    # -- Check 1: Count --
    a_files = sorted(sub.glob("*_a.pt"))
    b_files = sorted(sub.glob("*_b.pt"))
    other = [f for f in sub.iterdir()
             if not f.name.endswith("_a.pt") and not f.name.endswith("_b.pt")]

    if other:
        issues.append(f"Non-.pt files ({len(other)}): {[f.name for f in other[:10]]}")
    else:
        ok_parts.append("All files are .pt")

    if len(a_files) != n:
        issues.append(f"Type A: {len(a_files)} files, expected {n}")
    else:
        ok_parts.append(f"Type A: {len(a_files)} files OK")

    if len(b_files) != n:
        issues.append(f"Type B: {len(b_files)} files, expected {n}")
    else:
        ok_parts.append(f"Type B: {len(b_files)} files OK")

    # -- Check 2: Naming --
    a_indices = set()
    for f in a_files:
        try:
            a_indices.add(int(f.stem.replace("_a", "")))
        except ValueError:
            issues.append(f"Bad filename: {f.name}")

    b_indices = set()
    for f in b_files:
        try:
            b_indices.add(int(f.stem.replace("_b", "")))
        except ValueError:
            issues.append(f"Bad filename: {f.name}")

    expected = set(expected_indices)
    if a_indices != expected:
        mi = sorted(expected - a_indices)[:10]
        ex = sorted(a_indices - expected)[:10]
        if mi: issues.append(f"Type A missing indices: {mi}")
        if ex: issues.append(f"Type A extra indices: {ex}")
    else:
        ok_parts.append("Type A indices: complete")

    if b_indices != expected:
        mi = sorted(expected - b_indices)[:10]
        ex = sorted(b_indices - expected)[:10]
        if mi: issues.append(f"Type B missing indices: {mi}")
        if ex: issues.append(f"Type B extra indices: {ex}")
    else:
        ok_parts.append("Type B indices: complete")

    # -- Check 3: Loadable .pt --
    load_fails = []
    for f in a_files + b_files:
        try:
            t = torch.load(f, weights_only=True)
            if not isinstance(t, torch.Tensor):
                load_fails.append(f"{f.name}: not a tensor ({type(t).__name__})")
        except Exception as e:
            load_fails.append(f"{f.name}: load error - {e}")

    if load_fails:
        issues.append(f"Load errors ({len(load_fails)}):")
        issues.extend(f"  {x}" for x in load_fails[:10])
    else:
        ok_parts.append(f"Load check: all {len(a_files)+len(b_files)} .pt files OK")

    # -- Check 4: Shape match --
    shape_issues = []
    shape_ok = 0

    for idx in sorted(expected_indices):
        img_path = img / f"{idx:04d}.png"
        if not img_path.exists():
            continue

        try:
            with Image.open(img_path) as pil_img:
                w, h = pil_img.size
                expected_shape = (3, h, w)
        except Exception as e:
            issues.append(f"Cannot read {img_path}: {e}")
            continue

        for tag, suffix in [("a", f"{idx}_a.pt"), ("b", f"{idx}_b.pt")]:
            fpath = sub / suffix
            if not fpath.exists():
                continue
            try:
                t = torch.load(fpath, weights_only=True)
                actual = tuple(t.shape)
                if actual != expected_shape:
                    shape_issues.append(f"{suffix}: got {actual}, expected {expected_shape}")
                else:
                    shape_ok += 1
            except Exception:
                pass

    if shape_issues:
        issues.append(f"Shape mismatches ({len(shape_issues)}):")
        issues.extend(f"  {x}" for x in shape_issues[:20])
        if len(shape_issues) > 20:
            issues.append(f"  ... and {len(shape_issues)-20} more")
    else:
        ok_parts.append(f"Shape match: all {shape_ok} tensors OK")

    # -- Report --
    ok = len(issues) == 0
    lines = []
    lines.append("=" * 60)
    lines.append(f"Submission: {sub}")
    lines.append(f"Images:     {img}")
    lines.append("=" * 60)

    if ok_parts:
        lines.append("\n[PASS]")
        for pt in ok_parts:
            lines.append(f"  + {pt}")

    if issues:
        lines.append(f"\n[FAIL] {len(issues)} issue(s):")
        for i, issue in enumerate(issues, 1):
            lines.append(f"  {i}. {issue}")
    else:
        lines.append("\n[RESULT] ALL CHECKS PASSED - Submission is valid!")

    lines.append("=" * 60)
    return ok, "\n".join(lines)


In [10]:
import tempfile
from pathlib import Path

# Materialize the perturbations to a temp folder and run the checks against the
# test split's images (this is the folder layout `check` expects).
_pt_dir = Path(tempfile.mkdtemp())
for (delta_a, delta_b), item in zip(test_preds, test_targets):
    idx = item["idx"]
    torch.save(delta_a.contiguous(), _pt_dir / f"{idx}_a.pt")
    torch.save(delta_b.contiguous(), _pt_dir / f"{idx}_b.pt")

_images_dir = DATASET_ROOT / TEST_SPLIT / "images"
ok, report = check(_pt_dir, _images_dir)
print(report)
assert ok, "Submission checks failed — fix the issues above before building the zip."


Submission: /tmp/tmp8qiufy_v
Images:     dataset/public/test_public/images

[PASS]
  + Images found: 100 (dataset/public/test_public/images)
  + All files are .pt
  + Type A: 100 files OK
  + Type B: 100 files OK
  + Type A indices: complete
  + Type B indices: complete
  + Load check: all 200 .pt files OK
  + Shape match: all 200 tensors OK

[RESULT] ALL CHECKS PASSED - Submission is valid!


## 8. Evaluate locally

Before packaging, run the official scoring formula locally using the
already-loaded models. This is the same logic the grader uses — it verifies
each perturbation against both models and computes:

```
Pure  = (Score_A + Score_B) / (2M)
PF    = 1.5 - 1/(1 + exp(-100000 * mean_norm))
Final = Pure * PF
```

Run this on both `train` and `test_public` to check your scores before submitting.

In [11]:
import math
from scipy.special import expit

K = 100_000

def compute_score(preds, targets, verbose=True):
    """Official scorer — mirrors evaluate.py exactly."""
    score_a = score_b = 0
    l2_list = []
    total = len(targets)

    with torch.no_grad():
        for (delta_a, delta_b), item in zip(preds, targets):
            img = Image.open(item["image_path"]).convert("RGB")
            img_t = TF.to_tensor(img).to(DEVICE)
            label = item["label"]

            da = delta_a.to(DEVICE)
            db = delta_b.to(DEVICE)

            if da.shape != img_t.shape or db.shape != img_t.shape:
                continue

            pixel_count = da.numel()
            l2_list.append(
                (da.norm(2).item() + db.norm(2).item()) / pixel_count / 2
            )

            # Type A: ResNet correct, ViT wrong
            adv_a = torch.clamp(img_t + da, 0, 1)
            inp_a = preprocess(adv_a).unsqueeze(0)
            if (
                resnet(inp_a).argmax(1).item() == label
                and vit(inp_a).argmax(1).item() != label
            ):
                score_a += 1

            # Type B: ViT correct, ResNet wrong
            adv_b = torch.clamp(img_t + db, 0, 1)
            inp_b = preprocess(adv_b).unsqueeze(0)
            if (
                resnet(inp_b).argmax(1).item() != label
                and vit(inp_b).argmax(1).item() == label
            ):
                score_b += 1

    pure_score = (score_a + score_b) / (total * 2)
    mean_norm = float(np.mean(l2_list)) if l2_list else 0.0
    pf = float(1.5 - expit(K * mean_norm))
    final_score = pure_score * pf

    if verbose:
        sep = "=" * 50
        print(f"\n{sep}")
        print(f"Local Evaluation (official formula)")
        print(f"{sep}")
        print(f"Total images       : {total}")
        print(f"Score A (R ok / V wrong) : {score_a}/{total}")
        print(f"Score B (V ok / R wrong) : {score_b}/{total}")
        print(f"Pure score         : {pure_score:.4f}")
        print(f"Mean L2/pixel      : {mean_norm:.6e}")
        print(f"Penalty factor     : {pf:.4f}")
        print(f"Final score        : {final_score:.4f}")
        print(f"{sep}")

    return final_score


# Evaluate on the train subset (quick check)
_ = compute_score(preds, targets)

# Evaluate on the full test split
test_score = compute_score(test_preds, test_targets)
print(f"\nTest S_final = {test_score:.4f}")
# 0.8580 / 0.8193 local validation (simple setup)
# pure adversarial loss: 0.9042 / 0.8656 local validation
# + 1e+02 weight_decay: 0.9087 / 0.8697 local validation
# weight decay scheduling (1e+03 first, after 10 steps, if resnet & vit predictions still match then remove weight_decay): 0.9404 / 0.9051



Local Evaluation (official formula)
Total images       : 20
Score A (R ok / V wrong) : 20/20
Score B (V ok / R wrong) : 19/20
Pure score         : 0.9750
Mean L2/pixel      : 1.422120e-06
Penalty factor     : 0.9645
Final score        : 0.9404

Local Evaluation (official formula)
Total images       : 100
Score A (R ok / V wrong) : 94/100
Score B (V ok / R wrong) : 95/100
Pure score         : 0.9450
Mean L2/pixel      : 1.692035e-06
Penalty factor     : 0.9578
Final score        : 0.9051

Test S_final = 0.9051


In [12]:
test_preds, test_targets = solver.run(lb_a_data)

Generating naive perturbations: 100%|██████████| 100/100 [01:39<00:00,  1.01it/s]


In [13]:
lb_a_score = compute_score(test_preds, test_targets)
# 0.9465 on LB A


Local Evaluation (official formula)
Total images       : 100
Score A (R ok / V wrong) : 99/100
Score B (V ok / R wrong) : 99/100
Pure score         : 0.9900
Mean L2/pixel      : 1.763465e-06
Penalty factor     : 0.9560
Final score        : 0.9465


In [19]:
lb_a_score

0.9464669995023344

In [14]:
test_preds, test_targets = solver.run(lb_b_data)

Generating naive perturbations: 100%|██████████| 100/100 [01:30<00:00,  1.10it/s]


In [15]:
lb_b_score = compute_score(test_preds, test_targets)
# 0.9402 on LB B


Local Evaluation (official formula)
Total images       : 100
Score A (R ok / V wrong) : 98/100
Score B (V ok / R wrong) : 99/100
Pure score         : 0.9850
Mean L2/pixel      : 1.824278e-06
Penalty factor     : 0.9545
Final score        : 0.9402


In [18]:
lb_b_score

0.9402013173090513

## 9. Build the submission zip

The deliverable is a single **`submission.zip`** of **flat-listed** `.pt` files named
`{index}_a.pt` / `{index}_b.pt` — no subdirectories. `{index}` is the image's order in the
split. Each tensor must match the **original image resolution** (`3 × H × W`).

The helper below writes the zip straight from the `(delta_a, delta_b)` predictions.


In [16]:
import io, zipfile

def make_submission_zip(preds, targets, out_path):
    with zipfile.ZipFile(out_path, "w", zipfile.ZIP_STORED) as zf:
        for (delta_a, delta_b), item in zip(preds, targets):
            idx = item["idx"]
            for tag, delta in (("a", delta_a), ("b", delta_b)):
                buf = io.BytesIO()
                torch.save(delta.contiguous(), buf)
                zf.writestr(f"{idx}_{tag}.pt", buf.getvalue())
    print(f"Wrote {out_path}  ({len(preds)} images × 2 tensors)")

make_submission_zip(test_preds, test_targets, "submission.zip")

Wrote submission.zip  (100 images × 2 tensors)
